In [14]:
import pymongo
import os                # Para leer archivos y carpetas
import hashlib           # Para calcular el hash SHA-256
from sentence_transformers import SentenceTransformer # Para generar embeddings

print("Bibliotecas importadas.")

# --- 1. Conexión al Replica Set ---
# Esta cadena de conexión es la clave.
# Apunta a los 3 puertos que abriste en tu máquina (localhost)
# y le dice a Pymongo el nombre de tu clúster (`replicaSet`).
cadena_conexion = "mongodb://localhost:30001,localhost:30002,localhost:30003/?replicaSet=my-replicaset"

try:
    # Intenta conectarse al clúster
    client = pymongo.MongoClient(cadena_conexion, serverSelectionTimeoutMS=5000)
    
    # client.server_info() fuerza una conexión para probarla.
    # Si esto falla, saltará al 'except'.
    client.server_info() 
    print("¡Conexión a MongoDB exitosa!")
    
    # --- 2. Apuntar a tu BD y Colección ---
    # La tarea especifica una BD "Política" y colección "Discursos" [cite: 418]
    db = client["Política"]
    collection = db["Discursos"]
    print(f"Conectado a la base de datos '{db.name}' y colección '{collection.name}'.")

    # --- 3. (Opcional) Limpiar la colección para pruebas ---
    # ADVERTENCIA: Descomenta la línea de abajo si quieres borrar todo
    # lo que hay en la colección antes de volver a insertar.
    # Útil para probar tu script varias veces.
    
    collection.delete_many({})
    print("Colección 'Discursos' limpiada (para pruebas).")

except pymongo.errors.ServerSelectionTimeoutError as err:
    print(f"Error de conexión: No se pudo conectar al clúster.")
    print("Asegúrate de que tus contenedores de Docker estén corriendo ('docker-compose up -d').")
    print(f"Detalle: {err}")

Bibliotecas importadas.
¡Conexión a MongoDB exitosa!
Conectado a la base de datos 'Política' y colección 'Discursos'.
Colección 'Discursos' limpiada (para pruebas).


In [15]:
# --- 4. Cargar el Modelo de Machine Learning ---
# Esto es una operación pesada y solo debe hacerse UNA VEZ.
# Por eso está en una celda separada, fuera del bucle de archivos.

# El PDF sugiere 'sentence-transformers' 
# 'paraphrase-multilingual-MiniLM-L12-v2' es un modelo excelente
# y ligero que funciona bien en español.
modelo_embedding = 'paraphrase-multilingual-MiniLM-L12-v2'

print(f"Cargando el modelo '{modelo_embedding}'...")
print("Esto puede tardar varios minutos la primera vez que se ejecuta, ya que lo descarga.")

# Esta línea carga el modelo en la memoria RAM
model = SentenceTransformer(modelo_embedding)

print("Modelo de embeddings cargado y listo.")

Cargando el modelo 'paraphrase-multilingual-MiniLM-L12-v2'...
Esto puede tardar varios minutos la primera vez que se ejecuta, ya que lo descarga.
Modelo de embeddings cargado y listo.


In [17]:
# --- 5. Definir la ruta a los discursos ---
ruta_corpus = "./corpus_discursos" 

# --- 6. Preparar los documentos únicos (usando un diccionario) ---
# En lugar de una lista, usamos un diccionario.
# La clave será el hash_id, el valor será el documento.
# Esto filtra duplicados automáticamente en Python.
documentos_unicos = {} 

# Contadores para saber qué está pasando
archivos_procesados = 0
archivos_duplicados_encontrados = 0

print(f"Empezando a procesar archivos en la carpeta: {ruta_corpus}")

# --- 7. Iterar sobre cada archivo en la carpeta ---
for nombre_archivo in os.listdir(ruta_corpus):
    if nombre_archivo.endswith(".txt"):
        archivos_procesados += 1
        ruta_completa = os.path.join(ruta_corpus, nombre_archivo)
        
        try:
            # --- A. Leer el contenido del archivo ---
            with open(ruta_completa, 'r', encoding='utf-8') as f:
                texto_completo = f.read()

            # --- B. Calcular Hash SHA-256 (Requerimiento 2.1) ---
            hash_id = hashlib.sha256(texto_completo.encode('utf-8')).hexdigest()

            # --- C. Generar Embedding (Requerimiento 2.2) ---
            embedding = model.encode(texto_completo)

            # --- D. Crear el documento (Requerimiento 2.3) ---
            documento = {
                "_id": hash_id,
                "texto": texto_completo,
                "embedding": embedding.tolist(),
                "filename": nombre_archivo
            }
            
            # --- E. Manejar el duplicado en Python ---
            if hash_id not in documentos_unicos:
                documentos_unicos[hash_id] = documento
            else:
                # Encontramos un hash duplicado, lo contamos pero no lo agregamos
                archivos_duplicados_encontrados += 1
                # (Opcional) puedes imprimir qué archivo es duplicado:
                # print(f"  -> Archivo duplicado omitido: {nombre_archivo}")

        except Exception as e:
            print(f"Error procesando el archivo {nombre_archivo}: {e}")

print(f"\n--- Resumen del Procesamiento ---")
print(f"Archivos .txt leídos: {archivos_procesados}")
print(f"Archivos duplicados encontrados (y omitidos): {archivos_duplicados_encontrados}")
print(f"Total de documentos ÚNICOS a insertar: {len(documentos_unicos)}")

# --- 8. Insertar el Lote ÚNICO en MongoDB ---
if documentos_unicos:
    
    # Convertimos los valores del diccionario en una lista
    lista_para_insertar = list(documentos_unicos.values())
    
    try:
        # Esta inserción ya no necesita 'ordered=False'
        # porque estamos seguros de que la lista es 100% única.
        collection.insert_many(lista_para_insertar)
        print(f"\n¡Éxito! Se insertaron {len(lista_para_insertar)} documentos únicos en 'Discursos'.")
    except Exception as e:
        # Si hay un error aquí, es por otra razón (ej. conexión)
        print(f"Error inesperado durante la inserción en lote: {e}")
else:
    print("No se encontraron documentos .txt en la carpeta para insertar.")

Empezando a procesar archivos en la carpeta: ./corpus_discursos

--- Resumen del Procesamiento ---
Archivos .txt leídos: 680
Archivos duplicados encontrados (y omitidos): 1
Total de documentos ÚNICOS a insertar: 679

¡Éxito! Se insertaron 679 documentos únicos en 'Discursos'.


In [8]:
# --- 9. Verificar la Inserción ---
# Contamos cuántos documentos hay ahora en la colección

try:
    conteo = collection.count_documents({})
    print(f"Verificación: La colección 'Discursos' ahora contiene {conteo} documentos.")

    # (Opcional) Mostrar un documento de ejemplo
    if conteo > 0:
        print("\n--- Ejemplo de un documento insertado ---")
        un_documento = collection.find_one({}, {"texto": 0}) # No mostrar el texto
        print(un_documento)

except Exception as e:
    print(f"Error al verificar: {e}")

Verificación: La colección 'Discursos' ahora contiene 679 documentos.

--- Ejemplo de un documento insertado ---
{'_id': 'e18c554e1328feea02c78d5b43a5866226dbcf22772152f1707dbb115e0da267', 'embedding': [0.1948472112417221, -0.2028965950012207, 0.17553584277629852, 0.12890420854091644, 0.2324349284172058, -0.0838870033621788, 0.15571445226669312, 0.14152446389198303, -0.21636810898780823, 0.20470334589481354, -0.03508381173014641, -0.08356489986181259, -0.31096982955932617, 0.1502603143453598, 0.22826747596263885, 0.0975043922662735, 0.05999908596277237, -0.2686906158924103, -0.23698300123214722, 0.060163743793964386, -0.09401587396860123, -0.09235688298940659, -0.2872217893600464, 0.12291140109300613, 0.054588086903095245, 0.09946280717849731, 0.11764229089021683, 0.16082462668418884, -0.07736743241548538, 0.014423801563680172, -0.19556796550750732, -0.06514549255371094, 0.2838205099105835, 0.2737066149711609, 0.01823730394244194, 0.30809512734413147, 0.2074410617351532, -0.25376251339

In [9]:
# Importamos numpy para el manejo eficiente de arrays (vectores)
import numpy as np

# Importamos la función específica de scikit-learn
from sklearn.metrics.pairwise import cosine_similarity

print("Bibliotecas para similitud (Numpy y Scikit-learn) importadas.")

Bibliotecas para similitud (Numpy y Scikit-learn) importadas.


In [18]:
# --- 1. Cargar Datos en Memoria ---
# (Asumimos que tu variable 'collection' de la Celda 1 sigue activa)

# Preparamos listas para guardar los datos que traigamos de la BD
lista_ids = []
lista_filenames = []
lista_textos_completos = []
lista_embeddings = []

print("Cargando todos los documentos desde MongoDB a la memoria RAM...")

# --- 2. Consultar la Base de Datos ---
# Hacemos un .find() para traer TODOS los documentos.
# Usamos un "projection" para traer solo los campos que necesitamos.
# Esto es mucho más eficiente que traer el documento entero.
try:
    cursor = collection.find(
        {},  # Un filtro vacío ({}) significa "traer todo"
        {
            "_id": 1,           # 1 significa "incluir este campo"
            "texto": 1,
            "embedding": 1,
            "filename": 1
        }
    )

    # --- 3. Iterar sobre los resultados ---
    for doc in cursor:
        lista_ids.append(doc['_id'])
        lista_filenames.append(doc['filename'])
        lista_textos_completos.append(doc['texto'])
        lista_embeddings.append(doc['embedding'])

    print(f"¡Carga completa! Se cargaron {len(lista_ids)} documentos.")

    # --- 4. Convertir a Matriz de Numpy (¡CRÍTICO!) ---
    # scikit-learn necesita los embeddings en un formato de matriz numpy
    if lista_embeddings:
        matriz_embeddings = np.array(lista_embeddings)
        print(f"Matriz de embeddings creada con forma: {matriz_embeddings.shape}")
    else:
        print("Error: No se encontraron embeddings en la base de datos.")

except Exception as e:
    print(f"Error al cargar datos de MongoDB: {e}")

Cargando todos los documentos desde MongoDB a la memoria RAM...
¡Carga completa! Se cargaron 679 documentos.
Matriz de embeddings creada con forma: (679, 384)


In [19]:
# --- 5. Definir la Función de Búsqueda ---
# (Asumimos que 'model', 'matriz_embeddings', 'lista_ids' y 
# 'lista_textos_snippets' están cargados de celdas anteriores)

# --- 5. Definir la Función de Búsqueda (Versión 2) ---

def encontrar_mejor_fragmento(texto_completo, consulta_texto):
    """
    Función simple para encontrar un fragmento relevante.
    Busca la primera palabra clave de la consulta en el texto.
    """
    # Tomar las palabras de la consulta, ignorando palabras comunes
    palabras_consulta = set(consulta_texto.lower().split())
    palabras_comunes = {'de', 'la', 'el', 'y', 'los', 'las', 'con', 'a', 'para', 'en', 'es', 'un'}
    palabras_clave = [p for p in palabras_consulta if p not in palabras_comunes]

    mejor_snippet = "No se encontró fragmento..."
    
    for palabra in palabras_clave:
        try:
            # Encontrar la primera aparición de una palabra clave
            idx = texto_completo.lower().find(palabra)
            if idx != -1:
                # Tomar 150 caracteres antes y 150 después
                inicio = max(idx - 150, 0)
                fin = min(idx + 150, len(texto_completo))
                mejor_snippet = "..." + texto_completo[inicio:fin] + "..."
                break # Nos quedamos con el primero que encontremos
        except:
            pass # Ignorar errores
            
    # Si no encontramos palabras clave, devolvemos el inicio
    if mejor_snippet == "No se encontró fragmento...":
        mejor_snippet = texto_completo[:300] + "..."
        
    return mejor_snippet

def buscar_similares(consulta_texto, top_k=5):
    """
    Recibe una consulta de texto y devuelve los top_k resultados
    más similares de la base de datos.
    """
    print(f"\n--- Buscando resultados para: '{consulta_texto}' ---")
    
    consulta_embedding = model.encode(consulta_texto)
    similitudes = cosine_similarity(
        consulta_embedding.reshape(1, -1),
        matriz_embeddings
    )
    scores = similitudes[0]
    top_k_indices = np.argsort(scores)[-top_k:][::-1]
    
    print(f"Mostrando los {top_k} resultados más similares:")
    
    for i in top_k_indices:
        # --- ¡AQUÍ ESTÁ LA MAGIA! ---
        
        # 1. Obtenemos el nombre del archivo (Problema 1 resuelto)
        nombre_archivo = lista_filenames[i]
        
        # 2. Obtenemos el texto completo (para el Problema 2)
        texto_completo = lista_textos_completos[i]
        
        # 3. Buscamos el fragmento relevante (Problema 2 resuelto)
        fragmento_relevante = encontrar_mejor_fragmento(texto_completo, consulta_texto)
        
        print(f"\n  Score: {scores[i]:.4f}")
        print(f"  Archivo: {nombre_archivo}") # <--- ¡SOLUCIÓN 1!
        print(f"  ID: {lista_ids[i]}")
        print(f"  Fragmento: {fragmento_relevante}") # <--- ¡SOLUCIÓN 2!


    

In [21]:
# --- 6. Probar la Función de Búsqueda ---
# Este es el resultado que debe quedar visible en tu notebook 

# Prueba 1: Buscar sobre un tema de los discursos (Drogas)
#buscar_similares("la droga mata los sueños de los jóvenes")

# Prueba 2: Buscar sobre otro tema (Transporte)
buscar_similares("transporte público y calidad de vida")

# Prueba 3: Buscar sobre otro tema (Economía)
#buscar_similares("acuerdo comercial con brasil y noruega")


--- Buscando resultados para: 'transporte público y calidad de vida' ---
Mostrando los 5 resultados más similares:

  Score: 0.6780
  Archivo: 93588.txt
  ID: 1a64b8937d29ef68f7586578c4abac99642fe9a23b862baf34577271df591543
  Fragmento: ...ivo de todo gobierno, y que nuestro Gobierno lo tiene en el corazón, es mejorar la calidad de todos nuestros compatriotas. Y, sin duda, el sistema de transporte público es un factor muy importante en la calidad de vida de nuestros compatriotas. Y el mejor ejemplo de ello es que el Transantiago, en l...

  Score: 0.6610
  Archivo: 165760.txt
  ID: 3698add4867a712b128a1c4eba996b9030ee25bffb5c3ab9d945d41f844c60d2
  Fragmento: ...abajadores que con sus manos, con su compromiso y con su esfuerzo están construyendo estas nuevas etapas del Metro de nuestro país.

 

El Sistema de Transporte es muy importante porque afecta directamente la calidad de vida de los ciudadanos. Un buen Sistema de Transporte ahorra tiempo, genera segu...

  Score: 0.6307
  Archiv

In [22]:
import time
from pymongo.read_preferences import ReadPreference

# (Asumimos que 'client' y 'collection' de la Celda 1 siguen activos)

# ID único para nuestra prueba
id_prueba = "prueba-consistencia-123"
doc_prueba = {"_id": id_prueba, "tipo": "prueba"}

print("--- 1. Limpiando prueba anterior (si existe) ---")
# Usamos 'client' (que escribe en el PRIMARIO)
try:
    collection.delete_one({"_id": id_prueba})
    print(f"Documento de prueba '{id_prueba}' limpiado.")
except Exception as e:
    print(f"Error limpiando: {e}")

print("\n--- 2. Insertando documento en el PRIMARIO ---")
# 'client.default' escribe en el PRIMARIO
collection.insert_one(doc_prueba)
print(f"Documento '{id_prueba}' insertado en el PRIMARIO.")

# Esperamos 1 segundo para que los datos se repliquen
print("...Esperando 1 segundo por la replicación...")
time.sleep(1)

# --- 3. Conexión ESPECIAL al SECUNDARIO ---
print("\n--- 3. Conectando a un nodo SECUNDARIO (puerto 30002) ---")
try:
    # Creamos un nuevo cliente que se conecta a un secundario
    # y TIENE que ser parte del mismo replicaSet.
    client_secundario = pymongo.MongoClient(
        "mongodb://localhost:30002/?replicaSet=my-replicaset",
        read_preference=ReadPreference.SECONDARY
    )
    
    # Apuntamos a la misma colección
    collection_secundaria = client_secundario["Política"]["Discursos"]
    
    print("Conexión exitosa al SECUNDARIO.")

    # --- 4. Leer el documento desde el SECUNDARIO ---
    print(f"\n--- 4. Buscando '{id_prueba}' en el SECUNDARIO ---")
    
    doc_encontrado = collection_secundaria.find_one({"_id": id_prueba})
    
    if doc_encontrado:
        print("¡ÉXITO! El documento se replicó y fue encontrado:")
        print(doc_encontrado)
    else:
        print("FALLO. El documento no se encontró en el secundario.")
        
    client_secundario.close()

except Exception as e:
    print(f"Error conectando al secundario: {e}")

# --- 5. Limpieza final ---
finally:
    collection.delete_one({"_id": id_prueba})
    print(f"\n--- 5. Documento de prueba '{id_prueba}' eliminado del PRIMARIO. ---")

--- 1. Limpiando prueba anterior (si existe) ---
Documento de prueba 'prueba-consistencia-123' limpiado.

--- 2. Insertando documento en el PRIMARIO ---
Documento 'prueba-consistencia-123' insertado en el PRIMARIO.
...Esperando 1 segundo por la replicación...

--- 3. Conectando a un nodo SECUNDARIO (puerto 30002) ---
Conexión exitosa al SECUNDARIO.

--- 4. Buscando 'prueba-consistencia-123' en el SECUNDARIO ---
¡ÉXITO! El documento se replicó y fue encontrado:
{'_id': 'prueba-consistencia-123', 'tipo': 'prueba'}

--- 5. Documento de prueba 'prueba-consistencia-123' eliminado del PRIMARIO. ---


In [23]:
# --- Prueba ANTES del fallo ---
print("Ejecutando búsqueda en condiciones normales...")

# (Asumimos que la función 'buscar_similares' ya está definida)
buscar_similares("la droga y los jóvenes")

print("\nBúsqueda ANTES del fallo completada.")

Ejecutando búsqueda en condiciones normales...

--- Buscando resultados para: 'la droga y los jóvenes' ---
Mostrando los 5 resultados más similares:

  Score: 0.6945
  Archivo: 152358.txt
  ID: 553b9e8dea128da10514cc2981b9b47a2a3d69f029c3c55433bd3679f1e5c23f
  Fragmento: ...tra profunda preocupación por el alto consumo de drogas, como marihuana, cocaína, pasta base y tranquilizantes sin receta médica, de nuestros niños y jóvenes, y también por el inicio de este consumo a cada vez más temprana edad.

 

Como lo han ratificado una y otra vez las sociedades médicas y cien...

  Score: 0.5349
  Archivo: 93782.txt
  ID: b37c26fdfc60235667891321609e16abf4445e94aed0b202ff11b514aa7f3228
  Fragmento: ...ro por las astas, pero más vale tarde que nunca, porque lo cierto es que la droga está destruyendo a nuestras familias, a nuestros niños y a nuestros jóvenes.

 

Hoy día Chile tiene el triste récord de estar en el primer lugar entre todos los países de América en el consumo de marihuana, de coc

In [24]:
# --- Identificar al Primario ---
# (Asumimos 'client' de la Celda 1)
try:
    nodo_primario = client.primary
    print(f"El nodo PRIMARIO actual es: {nodo_primario}")
    print("Ve a tu terminal y detén el contenedor que corresponde a este puerto.")
except Exception as e:
    print(f"No se pudo encontrar al primario: {e}")

El nodo PRIMARIO actual es: ('host.docker.internal', 30001)
Ve a tu terminal y detén el contenedor que corresponde a este puerto.


In [25]:
# --- Prueba DURANTE el fallo ---
print("!!! El nodo PRIMARIO está DETENIDO. Ejecutando la misma búsqueda... !!!")
print("...Pymongo se 'congelará' por unos segundos mientras el clúster elige un nuevo primario...")

try:
    # Ejecutamos la MISMA búsqueda
    buscar_similares("la droga y los jóvenes")
    
    print("\n¡ÉXITO! La búsqueda funcionó.")
    print("Pymongo se reconectó automáticamente al nuevo primario.")

except Exception as e:
    print(f"\nFALLO. La búsqueda no funcionó: {e}")

!!! El nodo PRIMARIO está DETENIDO. Ejecutando la misma búsqueda... !!!
...Pymongo se 'congelará' por unos segundos mientras el clúster elige un nuevo primario...

--- Buscando resultados para: 'la droga y los jóvenes' ---
Mostrando los 5 resultados más similares:

  Score: 0.6945
  Archivo: 152358.txt
  ID: 553b9e8dea128da10514cc2981b9b47a2a3d69f029c3c55433bd3679f1e5c23f
  Fragmento: ...tra profunda preocupación por el alto consumo de drogas, como marihuana, cocaína, pasta base y tranquilizantes sin receta médica, de nuestros niños y jóvenes, y también por el inicio de este consumo a cada vez más temprana edad.

 

Como lo han ratificado una y otra vez las sociedades médicas y cien...

  Score: 0.5349
  Archivo: 93782.txt
  ID: b37c26fdfc60235667891321609e16abf4445e94aed0b202ff11b514aa7f3228
  Fragmento: ...ro por las astas, pero más vale tarde que nunca, porque lo cierto es que la droga está destruyendo a nuestras familias, a nuestros niños y a nuestros jóvenes.

 

Hoy día Chile tie

In [26]:
# --- Verificación Post-Fallo ---
print("Verificando el estado del clúster después de reiniciar el nodo caído...")
time.sleep(5) # Dar tiempo a que el nodo se una

try:
    info = client.server_info()
    print("¡Clúster saludable y reconectado!")
    print(f"El nuevo primario es: {client.primary}")
except Exception as e:
    print(f"Error: {e}")

Verificando el estado del clúster después de reiniciar el nodo caído...
¡Clúster saludable y reconectado!
El nuevo primario es: ('host.docker.internal', 30002)
